# squad-rag-bench on Kaggle (free)

**Before running:** in the right panel, set *Session options → Accelerator* to **GPU T4 x2** and *Internet* to **On**.
Then edit `REPO_URL` in cell 1 and run the cells in order (Shift+Enter). Full instructions: `RUN.md` in the repo.

GPU 0 runs the language model (Ollama). GPU 1 runs training, embeddings and the reranker.

## 1. Get the code

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/YOUR_USERNAME/squad-rag-bench.git"   # <- edit this

os.chdir("/kaggle/working")
if not os.path.exists("squad-rag-bench"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/squad-rag-bench")
print(os.getcwd())

## 2. Install and run the unit tests (expect `12 passed`)

In [ ]:
!pip install -q -e ".[dev]"
!python -m pytest -q

## 3. Check both GPUs are visible (expect two Tesla T4 rows)

In [ ]:
!nvidia-smi

## 4. Fine-tune the dense retriever on SQuAD 2.0 train (GPU 1)

In [ ]:
!CUDA_VISIBLE_DEVICES=1 python -m ragbench.train --config config_kaggle.yaml
!cat models/bge-small-squad/training_summary.json

## 5. Install Ollama, start it on GPU 0, download the language model (~5 GB)

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import time
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", OLLAMA_NUM_PARALLEL="4")
ollama_server = subprocess.Popen(["ollama", "serve"], env=env,
                                 stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
!ollama pull qwen2.5:7b

## 6. Smoke test on 40 questions (a few minutes). If this works, the full run will too.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 python -m ragbench.evaluate all --config config_kaggle.yaml --limit 40

## 7. Full benchmark: all retrieval questions + 500 generation questions per condition

In [ ]:
!CUDA_VISIBLE_DEVICES=1 python -m ragbench.evaluate all --config config_kaggle.yaml

## 8. Error analysis and demo-case selection

In [ ]:
!python -m ragbench.analyze --config config_kaggle.yaml

## 9. Demo: each demo question through every retriever, side by side

In [ ]:
!CUDA_VISIBLE_DEVICES=1 python -m ragbench.demo --config config_kaggle.yaml | tee results/demo_output.txt
# Ask your own question:
# !CUDA_VISIBLE_DEVICES=1 python -m ragbench.demo --config config_kaggle.yaml -q "Who ruled Normandy in the 10th century?" 

## 10. Package results for download (Output panel → results.zip)

In [ ]:
!cp models/bge-small-squad/training_summary.json results/
!zip -qr /kaggle/working/results.zip results
!ls -lh /kaggle/working/results.zip

**When finished:** Run → Stop Session. An open session uses your GPU quota even when idle.